In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from functools import reduce

from parreg.process_config import load_and_validate_config
from parreg import utils, utils_algo

import time

from parreg.logging_config import setup_logging
setup_logging()

import logging
logger = logging.getLogger(__name__)

In [2]:
# read and validate config
config_file = Path('/home/yuqiong.liu/work/Gitlab/ngen-regionalization/configs/config.yaml')
if not config_file.exists():
    raise FileNotFoundError(config_file)

config = load_and_validate_config(config_file)

2025-05-09 17:20:16,777 - parreg.process_config - INFO - Saving config to /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/config_final.yaml


In [3]:
# process by VPU
vpu = config.general.vpu_list[0]

In [4]:
# Check donors and receivers
donors_dict = config.donor.get_qualified_donors(vpu)

gdf = gpd.read_file(config.general.hydrofabric_file[vpu], layer = 'divides')
gdf_donors = gdf[gdf['divide_id'].isin(donors_dict['divide_id'])]
gdf_receivers = gdf[~gdf['divide_id'].isin(donors_dict['divide_id'])]
donors = gdf_donors['divide_id'].tolist()
receivers = gdf_receivers['divide_id'].tolist()
logger.info(f"Number of donors in vpu {vpu}: {len(donors)}")
logger.info(f"Number of receivers in vpu {vpu}: {len(receivers)}")

2025-05-09 17:20:16,888 - parreg.config_schema - INFO - Initial donors based on all gages in /home/yuqiong.liu/work/data/ngen_reg/gages_nwm4_calib_all.csv
2025-05-09 17:20:16,928 - parreg.config_schema - INFO - Number of initial donors for vpu 01: 88 gages, 4559 divides
2025-05-09 17:20:16,956 - parreg.config_schema - INFO - Number of donors after filtering: 77 gages, 3649 divides
2025-05-09 17:20:17,208 - __main__ - INFO - Number of donors in vpu 01: 3649
2025-05-09 17:20:17,209 - __main__ - INFO - Number of receivers in vpu 01: 16918


In [5]:
# compute the donor-receiver spatial distance
out = config.output.spatial_distance
dist_file = Path(out.path, 'donor_receiver_dist_' + config.general.domain + '_vpu' + vpu + '.' + out.format)
if dist_file.exists():
    logger.info(f"Spatial distance file already exists: {dist_file}\nSkip computing.")
    df_spatial_dist = utils.read_table(dist_file)
    #TODO: check if the spatial distance data includes all pairs of donors and receivers
    # if not, identify the missing pairs and compute the distance for them
else:
    logger.info(f'compute donor-receiver spatial distance ...')
    start_time = time.time()
    df_spatial_dist = utils_algo.compute_pairwise_centroid_distances(gdf_donors, gdf_receivers, 'divide_id', 'divide_id')
    end_time = time.time()
    print(f"Execution time: {end_time - start_time:.4f} seconds")

    # save the spatial distance data
    if out.save:
        if not dist_file.parent.is_dir():
            dist_file.parent.mkdir(parents=True, exist_ok=True)
        
        utils.save_data(df_spatial_dist, dist_file)
        logger.info(f"Spatial distance data saved to {dist_file}")  

2025-05-09 17:20:17,219 - __main__ - INFO - Spatial distance file already exists: /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/spatial_distance/donor_receiver_dist_conus_vpu01.parquet
Skip computing.


In [6]:
# process attribute data
datasets = config.general.attr_dataset_list
logger.info(f"Processing attribute data for VPU {vpu} ... datasets: {datasets}")

df_attrs_all = []
for dataset_name in datasets:

    dataset = getattr(config.attr_datasets, dataset_name)
    df_attrs = dataset.get_attr_data()
    
    df_attrs = df_attrs.rename(columns=lambda x: x if x == "divide_id" else f"{dataset_name}_{x}")

    # subset the attribute data to only include donors and receivers for the current VPU
    #TODO: add functionality to add additional donors from neighboring VPUs
    df_attrs = df_attrs[df_attrs['divide_id'].isin(donors + receivers)]

    df_attrs_all.append(df_attrs)

# Merge all attribute data frames column-wise, based on divide_id
df_attrs_all= reduce(
    lambda left, right: pd.merge(left, right, on='divide_id', how='outer'),
    df_attrs_all
)

# add a column to indicate whether the divide_id is a donor or receiver
df_attrs_all['is_donor'] = df_attrs_all['divide_id'].isin(donors)
# move the is_donor column to be the second column
df_attrs_all = df_attrs_all[['divide_id', 'is_donor'] + [col for col in df_attrs_all.columns if col not in ['divide_id', 'is_donor']]]

# check if all donors have attribute data
if not set(donors).issubset(df_attrs_all['divide_id']):
    logger.warning(f"Not all donors are included in the attribute data for VPU {vpu}.")
    missing_donors = [x for x in donors if x not in df_attrs_all['divide_id'].values]
    print("Missing donors:")
    print(missing_donors)

# check if all receivers have attribute data
if not set(receivers).issubset(df_attrs_all['divide_id']):
    logger.warning(f"Not all receivers are included in the attribute data for VPU {vpu}.")
    missing_receivers = [x for x in receivers if x not in df_attrs_all['divide_id'].values]
    print("Missing receivers:")
    print(missing_receivers)

# reset donor and receiver lists based on the attribute data
donors = df_attrs_all[df_attrs_all['is_donor']]['divide_id'].tolist()
receivers = df_attrs_all[~df_attrs_all['is_donor']]['divide_id'].tolist()

print(f'Number of donors in attribute data: {len(donors)}')
print(f'Number of receivers in attribute data: {len(receivers)}')

# check if all donors in attribute data are inlcuded in the donor_id of the spatial distance data
if not set(donors).issubset(df_spatial_dist['donor_id']):
    logger.warning(f"Not all donor_ids in the attribute data are present in the spatial distance data for VPU {vpu}.")
    missing_donor_ids = [x for x in donors if x not in df_spatial_dist['donor_id'].values]
    print("Missing donor_ids:")
    print(missing_donor_ids)

# check if all receivers in attribute data are inlcuded in the receiver_id of the spatial distance data
if not set(receivers).issubset(df_spatial_dist['receiver_id']):
    logger.warning(f"Not all receiver_ids in the attribute data are present in the spatial distance data for VPU {vpu}.")
    missing_receiver_ids = [x for x in receivers if x not in df_spatial_dist['receiver_id'].values]
    print("Missing receiver_ids:")
    print(missing_receiver_ids)

# sort the attribute data by is_donor and divide_id
df_attrs_all = df_attrs_all.sort_values(by=['is_donor', 'divide_id'], ascending=[True, True])

# check percentage of missing data
df_missing = df_attrs_all.isna().mean()*100
if df_missing.sum()>0:
    logger.warning(f"There are missing data for attributes in vpu {vpu}")
    print("Missing data percentage for each attribute:")
    print(df_missing.loc[df_missing > 0])

# save the attribute data
out1 = config.output.attr_data_final
if out1.save:
    if not Path(out1.path).is_dir():
        Path(out1.path).mkdir(parents=True, exist_ok=True)
    out_file = Path(out1.path, 'attr_' + config.general.domain + '_vpu' + vpu + '.' + out1.format)

    logger.info(f"Saving attribute data to {out_file}")
    utils.save_data(df_attrs_all, out_file)

2025-05-09 17:20:20,512 - __main__ - INFO - Processing attribute data for VPU 01 ... datasets: ['ngen', 'hlr']


Number of donors in attribute data: 3649
Number of receivers in attribute data: 16918


2025-05-09 17:20:27,137 - __main__ - WARNING - There are missing data for attributes in vpu 01
2025-05-09 17:20:27,139 - __main__ - INFO - Saving attribute data to /home/yuqiong.liu/work/data/ngen_reg/outputs/run_ngen_hlr/attr_data_final/attr_conus_vpu01.parquet


Missing data percentage for each attribute:
ngen_dksat       0.213935
ngen_psisat      0.213935
hlr_AQPERMNEW    1.808723
hlr_TAVE         1.808723
hlr_PPT          1.808723
hlr_PET          1.808723
hlr_PMPE         1.808723
hlr_SAND         1.808723
dtype: float64
